# 🔄 Model 2: Conformer (Convolution-Augmented Transformer)

**Input:** `(batch, N_BANDS, n_channels)` — bands as sequence  
**Feature output:** 128-D penultimate vector → SVM

## 0. Mount Google Drive
Required since this notebook reads/writes files under `/content/drive/MyDrive/...`.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import subprocess
print(subprocess.run(['nvidia-smi'],capture_output=True,text=True).stdout or '⚠️ No GPU')

Wed Sep 30 00:23:57 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   48C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
!pip install scikit-learn openpyxl -q

In [4]:
import os,gc,time,numpy as np,pandas as pd
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow.keras import layers,models,Input,optimizers as opt_module

BAND_NAMES=["Delta","Theta","Alpha","Beta","Gamma"]

def load_dataset(p):
    # Read the file depending on its format (Excel or CSV)
    df = pd.read_excel(p) if p.endswith(".xlsx") else pd.read_csv(p)
    # Identify columns containing float values and convert them to float32
    # to reduce memory usage
    float_cols = df.select_dtypes(include=["float64"]).columns
    df[float_cols] = df[float_cols].astype(np.float32)
    # NaN guard: a handful of windows can contain NaN feature values (e.g.
    # near-zero-power channels in relative band-power calc). Left in, these
    # propagate NaN through the whole network from batch 1 on.
    n_nan = df.isna().any(axis=1).sum()
    if n_nan:
        print(f"    [nan guard] dropping {n_nan} rows with NaN features")
        df = df.dropna().reset_index(drop=True)
    return df

def normalize_tensors(Xtr, Xv, Xte):
    """Fit mean/std on TRAIN split only (per-feature), apply to all splits."""
    mean = Xtr.mean(axis=0, keepdims=True)
    std  = Xtr.std(axis=0, keepdims=True)
    Xtr  = (Xtr - mean) / (std + 1e-8)
    Xv   = (Xv  - mean) / (std + 1e-8)
    Xte  = (Xte - mean) / (std + 1e-8)
    return Xtr, Xv, Xte
def get_band_columns(df): return {b:[c for c in df.columns if c.startswith(f"{b}_")] for b in BAND_NAMES}
def prepare_tensors_seq(df):
    bc=get_band_columns(df)
    X=np.stack([df[bc[b]].values for b in BAND_NAMES],axis=1)  # (N,5,n_ch)
    return X, df["label"].values.astype(int)
def evaluate_with_svm(Xtr, ytr, Xte, yte, max_train=20000):
    # RBF-kernel SVC scales ~O(n^2) to O(n^3) in sample count -- subsample
    # for this evaluation step only (the 128-D deep feature embedding is
    # what's being evaluated, so a representative subsample suffices).
    if len(Xtr) > max_train:
        rng = np.random.RandomState(42)
        idx = rng.choice(len(Xtr), max_train, replace=False)
        Xtr, ytr = Xtr[idx], ytr[idx]
    sc = StandardScaler(); Xtr = sc.fit_transform(Xtr); Xte = sc.transform(Xte)
    svm = SVC(kernel="rbf", C=1.0, gamma="scale", class_weight="balanced", random_state=42)
    svm.fit(Xtr, ytr); p = svm.predict(Xte)

    tn, fp, fn, tp = confusion_matrix(yte, p, labels=[0, 1]).ravel()
    sensitivity = tp / (tp + fn) if (tp + fn) else 0.0
    specificity = tn / (tn + fp) if (tn + fp) else 0.0

    return {
        "accuracy": round(accuracy_score(yte, p), 4),
        "sensitivity": round(sensitivity, 4),
        "specificity": round(specificity, 4),
        "f1_macro": round(f1_score(yte, p, average="macro"), 4),
        "f1_weighted": round(f1_score(yte, p, average="weighted"), 4),
        "report": classification_report(yte, p, labels=[0, 1], target_names=["No Seizure", "Seizure"])
    }
def save_results(row,path):
    os.makedirs(os.path.dirname(path),exist_ok=True)
    pd.DataFrame([row]).to_csv(path,mode="a",header=not os.path.exists(path),index=False)
    print("Saved:",row)
def compile_model(m, opt):
    m.compile(optimizer=opt, loss="binary_crossentropy",
              metrics=["accuracy", tf.keras.metrics.AUC(name="auc"),
                       tf.keras.metrics.Precision(name="precision"),
                       tf.keras.metrics.Recall(name="recall")])
    return m


In [5]:
import numpy as np
import tensorflow as tf

# 1. Ensure reproducibility
SEED = 42
tf.random.set_seed(SEED)
np.random.seed(SEED)

# 2. Prevent GPU Out-Of-Memory (OOM) errors
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
    except RuntimeError as e:
        print(e)

In [6]:
DATASET_A  = "/content/drive/MyDrive/epilepsy_outputs/dataset_A.csv"
DATASET_B  = "/content/drive/MyDrive/epilepsy_outputs/dataset_B.csv"
RESULTS    = "/content/drive/MyDrive/epilepsy_outputs/results.csv"
OUTPUT_DIR = "/content/drive/MyDrive/epilepsy_outputs"
EPOCHS=50; BATCH_SIZE=64; PATIENCE=8; SEED=42
MODEL_NAME = "conformer"

## Architecture

```
Input -> Dense projection (d_model)
-> N × ConformerBlock:
     FeedForward (0.5 weight) -> MHSA -> DepthwiseConv -> FeedForward
-> GlobalAvgPool -> Dense(128, ReLU) ← feature vector -> Dense(1, sigmoid)
```

In [7]:
# ── Conformer building blocks ────────────────────────────────────────────────
class FeedForwardModule(layers.Layer):
    def __init__(self,d,ff=4,dr=0.1,**kw):
        super().__init__(**kw)
        self.ln=layers.LayerNormalization()
        self.ff1=layers.Dense(d*ff,activation="swish")
        self.dr1=layers.Dropout(dr)
        self.ff2=layers.Dense(d)
        self.dr2=layers.Dropout(dr)
    def call(self,x,training=False):
        r=x; x=self.ln(x); x=self.ff1(x); x=self.dr1(x,training=training)
        x=self.ff2(x); x=self.dr2(x,training=training); return 0.5*x+r

class ConvolutionModule(layers.Layer):
    def __init__(self,d,k=3,dr=0.1,**kw):
        super().__init__(**kw)
        self.ln=layers.LayerNormalization()
        self.pw1=layers.Conv1D(d*2,1)
        self.dw=layers.DepthwiseConv1D(k,padding="same",use_bias=False)
        self.bn=layers.BatchNormalization()
        self.pw2=layers.Conv1D(d,1)
        self.dr=layers.Dropout(dr)
    def call(self,x,training=False):
        # import tensorflow as tf
        r=x; x=self.ln(x); x=self.pw1(x)
        x1,x2=tf.split(x,2,axis=-1); x=x1*tf.nn.sigmoid(x2)
        x=self.dw(x); x=self.bn(x,training=training); x=tf.nn.swish(x)
        x=self.pw2(x); x=self.dr(x,training=training); return x+r

class ConformerBlock(layers.Layer):
    def __init__(self,d,h=4,k=3,dr=0.1,**kw):
        super().__init__(**kw)
        self.ff1=FeedForwardModule(d,dr=dr)
        self.ln_sa=layers.LayerNormalization()
        self.mhsa=layers.MultiHeadAttention(num_heads=h,key_dim=d//h,dropout=dr)
        self.dr_sa=layers.Dropout(dr)
        self.conv=ConvolutionModule(d,k,dr)
        self.ff2=FeedForwardModule(d,dr=dr)
        self.ln=layers.LayerNormalization()
    def call(self,x,training=False):
        x=self.ff1(x,training=training)
        r=x; xn=self.ln_sa(x); x=self.mhsa(xn,xn,training=training)
        x=self.dr_sa(x,training=training); x=x+r
        x=self.conv(x,training=training); x=self.ff2(x,training=training)
        return self.ln(x)

def build_conformer(n_bands=5,n_channels=18,d_model=64,num_heads=4,
                    num_blocks=4,feature_dim=128,dropout=0.1,output_bias=None):
    bias_init = tf.keras.initializers.Constant(output_bias) if output_bias is not None else "zeros"
    inp=Input(shape=(n_bands,n_channels),name="eeg_input")
    x=layers.Dense(d_model,name="proj")(inp)
    for i in range(num_blocks):
        # x=ConformerBlock(d_model,num_heads,dropout_rate=dropout,name=f"cb_{i}")(x)
        # Change dropout_rate to dr
        x = ConformerBlock(d_model, num_heads, dr=dropout, name=f"cb_{i}")(x)
    x=layers.GlobalAveragePooling1D()(x)
    feat=layers.Dense(feature_dim,activation="relu",name="feature_vector")(x)
    feat=layers.Dropout(dropout)(feat)
    out=layers.Dense(1,activation="sigmoid",name="output",bias_initializer=bias_init)(feat)
    return models.Model(inp,out,name="Conformer"),models.Model(inp,feat,name="Conformer_Feat")


In [8]:
model, feat_ext = build_conformer()
model.summary()
print("Feature extractor output:", feat_ext.output_shape)

# Clean up the dummy graph before starting the real training loop
del model, feat_ext
tf.keras.backend.clear_session()

Model: "Conformer"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ eeg_input (InputLayer)          │ (None, 5, 18)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ proj (Dense)                    │ (None, 5, 64)          │         1,216 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ cb_0 (ConformerBlock)           │ (None, 5, 64)          │        96,384 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ cb_1 (ConformerBlock)           │ (None, 5, 64)          │        96,384 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ cb_2 (ConformerBlock)           │ (None, 5, 64)          │        96,384 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ cb_3 (ConformerBlock)           │ (None, 5, 64)          │        96,384 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d        │ (None, 64)             │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ feature_vector (Dense)          │ (None, 128)            │         8,320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_28 (Dropout)            │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ output (Dense)                  │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 395,201 (1.51 MB)

 Trainable params: 394,689 (1.51 MB)

 Non-trainable params: 512 (2.00 KB)

Feature extractor output: (None, 128)


In [9]:
def run_trial(ds_path, ds_label, opt_name, opt_obj):
    print(f"\n{'='*55}\n  {MODEL_NAME} | Dataset {ds_label} | {opt_name}\n{'='*55}")
    df=load_dataset(ds_path); X,y=prepare_tensors_seq(df)
    n_bands,n_ch=X.shape[1],X.shape[2]
    Xtrv,Xte,ytrv,yte=train_test_split(X,y,test_size=0.15,stratify=y,random_state=SEED)
    Xtr,Xv,ytr,yv=train_test_split(Xtrv,ytrv,test_size=0.15/0.85,stratify=ytrv,random_state=SEED)
    Xtr,Xv,Xte=normalize_tensors(Xtr,Xv,Xte)   # train-only normalization
    total=len(ytr); n1=ytr.sum(); cw={0:total/(2*(total-n1)),1:total/(2*n1)}
    # Milder bias than CNN2D's: an extreme negative bias (~-6.3 here) combined
    # with this model's LayerNorm-heavy attention stack causes a training
    # collapse (AUC pinned at exactly 0.5, precision/recall=0 for 9+ epochs
    # straight -- confirmed via a live run). Cap the bias magnitude so the
    # sigmoid starts in a less saturated region.
    output_bias=np.clip(np.log(n1/(total-n1)), -3.0, 0.0)
    full_m,feat_m=build_conformer(n_bands=n_bands,n_channels=n_ch,output_bias=output_bias)
    full_m=compile_model(full_m,opt_obj)
    ckpt=os.path.join(OUTPUT_DIR,"checkpoints",f"{MODEL_NAME}_{ds_label}_{opt_name}_best.keras")
    os.makedirs(os.path.dirname(ckpt),exist_ok=True)
    cbs=[tf.keras.callbacks.EarlyStopping(monitor="val_auc",mode="max",patience=PATIENCE,restore_best_weights=True,verbose=1),
         tf.keras.callbacks.ModelCheckpoint(ckpt,monitor="val_auc",mode="max",save_best_only=True,verbose=0),
         tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss",factor=0.5,patience=4,min_lr=1e-6,verbose=1)]
    t0=time.time()
    hist=full_m.fit(Xtr,ytr,validation_data=(Xv,yv),epochs=EPOCHS,batch_size=BATCH_SIZE,
                    class_weight=cw,callbacks=cbs,verbose=1)
    elapsed=round((time.time()-t0)/60,2)
    best_auc=max(hist.history.get("val_auc",[0]))
    print(f"  Done {elapsed}min | Best AUC:{best_auc:.4f}")
    f_tr=feat_m.predict(Xtr,batch_size=BATCH_SIZE,verbose=0)
    f_te=feat_m.predict(Xte,batch_size=BATCH_SIZE,verbose=0)
    res=evaluate_with_svm(f_tr,ytr,f_te,yte)
    print(f"  SVM Acc:{res['accuracy']} F1:{res['f1_macro']}\n",res["report"])
    save_results({"model":MODEL_NAME,"dataset":f"Dataset_{ds_label}","optimizer":opt_name,
                  "best_val_auc":round(best_auc,4),**{k:v for k,v in res.items() if k!="report"},
                  "train_time_min":elapsed},RESULTS)
    del full_m,feat_m; gc.collect(); tf.keras.backend.clear_session()

for ds_label,ds_path in [("A",DATASET_A),("B",DATASET_B)]:
    # Lower LR (was 1e-3): attention stacks are more sensitive to LR than
    # plain conv nets, and 1e-3 combined with the bias-init above may still
    # be too aggressive for the first few epochs.
    for opt_name,opt_obj in {"Adam":opt_module.Adam(2e-4,clipnorm=1.0),
                              "Nadam":opt_module.Nadam(2e-4,clipnorm=1.0),
                              "AdamW":opt_module.AdamW(2e-4,weight_decay=1e-4,clipnorm=1.0)}.items():
        run_trial(ds_path,ds_label,opt_name,opt_obj)
print(f"\n✅ {MODEL_NAME} all trials done.")



  conformer | Dataset A | Adam
    [nan guard] dropping 7 rows with NaN features
Epoch 1/50
7560/7560 ━━━━━━━━━━━━━━━━━━━━ 191s 17ms/step - accuracy: 0.9979 - auc: 0.6705 - loss: 1.3237 - precision: 0.0000e+00 - recall: 0.0000e+00 - val_accuracy: 0.9980 - val_auc: 0.4995 - val_loss: 0.0929 - val_precision: 0.0000e+00 - val_recall: 0.0000e+00 - learning_rate: 2.0000e-04
Epoch 2/50
7560/7560 ━━━━━━━━━━━━━━━━━━━━ 86s 11ms/step - accuracy: 0.9980 - auc: 0.5965 - loss: 1.2352 - precision: 0.0000e+00 - recall: 0.0000e+00 - val_accuracy: 0.9980 - val_auc: 0.4874 - val_loss: 0.1056 - val_precision: 0.0000e+00 - val_recall: 0.0000e+00 - learning_rate: 2.0000e-04
Epoch 3/50
7560/7560 ━━━━━━━━━━━━━━━━━━━━ 85s 11ms/step - accuracy: 0.9980 - auc: 0.5129 - loss: 1.1559 - precision: 0.0000e+00 - recall: 0.0000e+00 - val_accuracy: 0.9980 - val_auc: 0.5000 - val_loss: 0.1411 - val_precision: 0.0000e+00 - val_recall: 0.0000e+00 - learning_rate: 2.0000e-04
Epoch 4/50
7560/7560 ━━━━━━━━━━━━━━━━━━━━ 86s 1